# 03 — Tokenization & BIO Label-Alignment Pipeline
### Cross-Domain NER Capstone (AAI590, Group 1)

This is the piece every downstream model depends on: the baseline, few-shot fine-tuning, and DAP fine-tuning all need their text converted the same way before they can train.



BERT doesn't read whole words — it reads **subword pieces**. A word like `"Ostendorf"` might get split into `["O", "##sten", "##dorf"]`. That means the tag list above no longer lines up token-for-token with what BERT actually sees, and training will silently break (or worse, train on misaligned labels) unless we fix the alignment ourselves. That's the whole job of this notebook.



In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2"
print("Done installing.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
Done installing.


## Step 1 — Load the processed data




In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
PROCESSED = Path('/content/drive/MyDrive/AAI590/data/processed')
FEWSHOT = PROCESSED / 'fewshot_splits'

for name in ['conll2003', 'wnut17', 'scierc']:
    d = PROCESSED / name
    print(name, '->', sorted(p.name for p in d.glob('*')) if d.exists() else 'NOT FOUND YET')

print('\nfewshot_splits ->', sorted(p.name for p in FEWSHOT.glob('*')) if FEWSHOT.exists() else 'NOT FOUND YET')


Mounted at /content/drive
conll2003 -> ['conll2003_test.jsonl', 'conll2003_train.jsonl', 'conll2003_validation.jsonl', 'summary.json']
wnut17 -> ['summary.json', 'wnut17_test.jsonl', 'wnut17_train.jsonl', 'wnut17_validation.jsonl']
scierc -> ['scierc_test.jsonl', 'scierc_train.jsonl', 'scierc_validation.jsonl', 'summary.json']

fewshot_splits -> ['manifest.csv', 'scierc', 'wnut17']


In [3]:
import json

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_full_split(dataset_name, split):
    fp = PROCESSED / dataset_name / f"{dataset_name}_{split}.jsonl"
    return load_jsonl(fp) if fp.exists() else []

# Sanity check: load CoNLL-2003 train and look at one example
conll_train_raw = load_full_split("conll2003", "train")
print("CoNLL-2003 train sentences:", len(conll_train_raw))
print("Example:", conll_train_raw[0])


CoNLL-2003 train sentences: 14041
Example: {'id': 'train-0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'tags': ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O'], 'source_dataset': 'conll2003', 'split': 'train'}


## Step 2 — Build a label list for each dataset

Each dataset has its own entity inventory (CoNLL-2003 uses 4 types, WNUT-17 and SciERC each use 6 different types), so each one needs its **own** label list and its own `label2id` / `id2label` mapping. We scan every split of a dataset (train + validation + test) to make sure no tag gets missed, then save these mappings to Drive so every later notebook uses the exact same label numbering.


In [4]:
def build_label_list(dataset_name):
    tags = set()
    for split in ("train", "validation", "test"):
        for row in load_full_split(dataset_name, split):
            tags.update(row["tags"])
    # Put "O" first by convention, then sort the rest for a stable, reproducible ordering
    label_list = ["O"] + sorted(t for t in tags if t != "O")
    return label_list

label_lists = {}
for ds in ["conll2003", "wnut17", "scierc"]:
    label_lists[ds] = build_label_list(ds)
    print(ds, "->", len(label_lists[ds]), "labels:", label_lists[ds])


conll2003 -> 8 labels: ['O', 'B-LOC', 'B-MISC', 'B-ORG', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER']
wnut17 -> 13 labels: ['O', 'B-corporation', 'B-creative-work', 'B-group', 'B-location', 'B-person', 'B-product', 'I-corporation', 'I-creative-work', 'I-group', 'I-location', 'I-person', 'I-product']
scierc -> 13 labels: ['O', 'B-Generic', 'B-Material', 'B-Method', 'B-Metric', 'B-OtherScientificTerm', 'B-Task', 'I-Generic', 'I-Material', 'I-Method', 'I-Metric', 'I-OtherScientificTerm', 'I-Task']


In [5]:
label_maps = {}
for ds, labels in label_lists.items():
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for i, l in enumerate(labels)}
    label_maps[ds] = {"label2id": label2id, "id2label": id2label}

# Save to Drive so training notebooks load the *same* mapping instead of rebuilding it
LABELS_DIR = PROCESSED / "label_maps"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
for ds, maps in label_maps.items():
    with open(LABELS_DIR / f"{ds}_label_map.json", "w") as f:
        json.dump(maps, f, indent=2)

print("Saved label maps to:", LABELS_DIR)


Saved label maps to: /content/drive/MyDrive/AAI590/data/processed/label_maps


## Step 3 — Load the tokenizer and check subword splitting

Checking what `bert-base-cased` does to a sentence, so the alignment code isn't a black box.


In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example_tokens = conll_train_raw[0]["tokens"]
example_tags = conll_train_raw[0]["tags"]

encoding = tokenizer(example_tokens, is_split_into_words=True)
subwords = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

print("Original words: ", example_tokens)
print("Original tags:  ", example_tags)
print("\nBERT subwords:  ", subwords)
print("word_ids():     ", encoding.word_ids())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original words:  ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Original tags:   ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O']

BERT subwords:   ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
word_ids():      [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Step 4 — Write the label-alignment function

The standard convention (used in the Hugging Face token-classification: **only the first subword of each word keeps the real tag.** Every other subword piece from the same word gets a special ignore value (`-100`), which tells PyTorch's loss function to skip it entirely during training. `[CLS]`, `[SEP]`, and padding tokens also get `-100`.

Why not just repeat the tag across every subword piece instead? Because for a `B-ORG` word split into 3 pieces, repeating `B-ORG` three times would tell the model three separate entities are *beginning* at once, which is wrong — there's still only one entity. Marking only the first piece keeps the label meaning correct.


In [7]:
def tokenize_and_align_labels(tokens, tags, tokenizer, label2id):
    encoding = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=256)
    word_ids = encoding.word_ids()

    aligned_labels = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            # Special token ([CLS], [SEP], padding)
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            # First subword of a new word -- keep the real label
            aligned_labels.append(label2id[tags[word_id]])
        else:
            # A later subword piece of the same word -- ignore in the loss
            aligned_labels.append(-100)
        prev_word_id = word_id

    encoding["labels"] = aligned_labels
    return encoding

# Try it on the same example from Step 3
label2id = label_maps["conll2003"]["label2id"]
result = tokenize_and_align_labels(example_tokens, example_tags, tokenizer, label2id)

print("Subwords:        ", tokenizer.convert_ids_to_tokens(result["input_ids"]))
print("Aligned label ids:", result["labels"])
id2label = label_maps["conll2003"]["id2label"]
print("Aligned labels:   ", [id2label.get(i, "IGNORE(-100)") if i != -100 else "IGNORE(-100)" for i in result["labels"]])


Subwords:         ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
Aligned label ids: [-100, 6, 0, 5, 0, 0, 0, 5, 0, -100, 0, -100]
Aligned labels:    ['IGNORE(-100)', 'I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'IGNORE(-100)', 'O', 'IGNORE(-100)']


## Step 5 — Apply this to a full dataset split and save it

Now scale the function above up to an entire split (all sentences, not just one example), using Hugging Face's `datasets` library so we get batched processing and an easy `.save_to_disk(...)` for reuse in the training notebooks.


In [8]:
from datasets import Dataset

def build_tokenized_dataset(rows, dataset_name, tokenizer):
    label2id = label_maps[dataset_name]["label2id"]

    def process_batch(batch):
        all_input_ids, all_attention_mask, all_labels = [], [], []
        for tokens, tags in zip(batch["tokens"], batch["tags"]):
            enc = tokenize_and_align_labels(tokens, tags, tokenizer, label2id)
            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_labels.append(enc["labels"])
        return {
            "input_ids": all_input_ids,
            "attention_mask": all_attention_mask,
            "labels": all_labels,
        }

    raw_ds = Dataset.from_list(rows)
    tokenized_ds = raw_ds.map(process_batch, batched=True, remove_columns=raw_ds.column_names)
    return tokenized_ds

# Quick test on CoNLL-2003 validation (smaller than train, faster to sanity-check)
conll_val_raw = load_full_split("conll2003", "validation")
conll_val_tokenized = build_tokenized_dataset(conll_val_raw, "conll2003", tokenizer)
print(conll_val_tokenized)
print("\nFirst example:", conll_val_tokenized[0])


Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3250
})

First example: {'input_ids': [101, 15531, 9741, 22441, 1942, 118, 149, 27514, 10954, 9272, 9637, 1708, 3048, 18172, 2036, 157, 1592, 22441, 152, 17145, 2069, 13020, 16972, 2101, 138, 26321, 9637, 15969, 27451, 11780, 1708, 7118, 16647, 9565, 3663, 119, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, -100, -100, -100, 0, 6, -100, -100, -100, -100, -100, -100, -100, -100, 0, -100, -100, 0, -100, -100, 0, 0, -100, 0, -100, -100, 0, -100, -100, -100, 0, -100, -100, -100, 0, -100]}


## Step 6 — Process everything the training will need, and save to Drive

This covers three things:
1. **CoNLL-2003** — all three splits, for baseline training and in-domain evaluation.
2. **WNUT-17 and SciERC test sets** — needed to evaluate any model (baseline or adapted) on the target domains.
3. **Every few-shot split** (50/100/200 labels × 3 seeds, for WNUT-17 and SciERC) — needed for fine-tuning and DAP experiments.



In [9]:
TOKENIZED_DIR = PROCESSED / "tokenized"
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. CoNLL-2003: all splits ---
for split in ["train", "validation", "test"]:
    rows = load_full_split("conll2003", split)
    ds = build_tokenized_dataset(rows, "conll2003", tokenizer)
    out_path = TOKENIZED_DIR / "conll2003" / split
    ds.save_to_disk(str(out_path))
    print(f"Saved conll2003/{split}: {len(ds)} examples -> {out_path}")


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saved conll2003/train: 14041 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/conll2003/train


Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saved conll2003/validation: 3250 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/conll2003/validation


Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

Saved conll2003/test: 3453 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/conll2003/test


In [10]:
# --- 2. WNUT-17 and SciERC: full validation + test splits ---
for ds_name in ["wnut17", "scierc"]:
    for split in ["validation", "test"]:
        rows = load_full_split(ds_name, split)
        if not rows:
            print(f"  (skipping {ds_name}/{split} -- not found)")
            continue
        ds = build_tokenized_dataset(rows, ds_name, tokenizer)
        out_path = TOKENIZED_DIR / ds_name / split
        ds.save_to_disk(str(out_path))
        print(f"Saved {ds_name}/{split}: {len(ds)} examples -> {out_path}")


Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1009 [00:00<?, ? examples/s]

Saved wnut17/validation: 1009 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/wnut17/validation


Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1287 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/275 [00:00<?, ? examples/s]

Saved scierc/validation: 275 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/scierc/validation


Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/551 [00:00<?, ? examples/s]

Saved scierc/test: 551 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/scierc/test


In [11]:
# --- 3. Every few-shot split (50/100/200 x 3 seeds) for WNUT-17 and SciERC ---
BUDGETS = [50, 100, 200]
SEEDS = [13, 42, 101]

for ds_name in ["wnut17", "scierc"]:
    for budget in BUDGETS:
        for seed in SEEDS:
            # Matches the exact naming used by 02_make_fewshot_splits.ipynb
            fp = FEWSHOT / ds_name / f"{ds_name}_train_{budget}_seed_{seed}.jsonl"
            if not fp.exists():
                print(f"  (skipping, not found: {fp.name})")
                continue
            rows = load_jsonl(fp)
            ds = build_tokenized_dataset(rows, ds_name, tokenizer)
            out_path = TOKENIZED_DIR / "fewshot" / ds_name / f"budget{budget}_seed{seed}"
            ds.save_to_disk(str(out_path))
            print(f"Saved {ds_name} budget={budget} seed={seed}: {len(ds)} examples -> {out_path}")


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saved wnut17 budget=50 seed=13: 50 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/fewshot/wnut17/budget50_seed13


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saved scierc budget=200 seed=42: 200 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/fewshot/scierc/budget200_seed42


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saved scierc budget=200 seed=101: 200 examples -> /content/drive/MyDrive/AAI590/data/processed/tokenized/fewshot/scierc/budget200_seed101
